# SPARK Pipeline — Google Colab A100/H100
Exécuter les cellules dans l'ordre. Prévoir ~15-20 min pour un run complet.

## 0. Vérification GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
# Doit afficher A100 ou H100 — sinon: Runtime > Change runtime type > A100

## 1. Cloner le repo

In [ ]:
import os
REPO_URL = 'https://github.com/Blockprod/SPARK.git'
PROJECT_DIR = '/content/SPARK'
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    # Écraser les modifications locales (config.yaml réécrit par cell 5)
    # pour garantir un pull propre à chaque relance
    !git -C {PROJECT_DIR} checkout -- .
    !git -C {PROJECT_DIR} pull
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())


## 2. Installer les dépendances

In [ ]:
# Forcer transformers==4.46.3 (Colab installe une version trop récente incompatible avec diffusers/LTX)
!pip install -q \
    pydantic-settings PyYAML python-dotenv httpx tenacity \
    google-genai \
    'diffusers>=0.32.0,<0.34.0' transformers==4.46.3 accelerate \
    safetensors sentencepiece \
    kokoro-onnx==0.4.9 onnxruntime-gpu soundfile librosa \
    ffmpeg-python pysubs2 \
    pytrends praw \
    google-api-python-client google-auth-httplib2 google-auth-oauthlib \
    APScheduler uvicorn sse-starlette jinja2 python-multipart \
    rich cryptography

!apt-get install -qq ffmpeg
print('Installation terminée.')

## 3. Monter Google Drive et copier les secrets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SECRETS = '/content/drive/MyDrive/SPARK_secrets'
import shutil, os

shutil.copy(f'{DRIVE_SECRETS}/.env', '/content/SPARK/.env')
os.makedirs('/content/SPARK/secrets', exist_ok=True)
shutil.copy(f'{DRIVE_SECRETS}/client_secret.json', '/content/SPARK/secrets/client_secret.json')
shutil.copy(f'{DRIVE_SECRETS}/youtube_token.json', '/content/SPARK/secrets/youtube_token.json')
os.makedirs('/content/SPARK/models', exist_ok=True)
shutil.copy(f'{DRIVE_SECRETS}/models/kokoro-v1.0.fp16.onnx', '/content/SPARK/models/kokoro-v1.0.fp16.onnx')
shutil.copy(f'{DRIVE_SECRETS}/models/voices-v1.0.bin', '/content/SPARK/models/voices-v1.0.bin')
print('Secrets copiés ✓')

## 4. Connexion HuggingFace (pour LTX-Video)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('HuggingFace connecté ✓')

## 5. Config A100 — qualité maximale

In [ ]:
import yaml
config_path = '/content/SPARK/config.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# ── Activer LTX-Video (provider "static" = écran noir !) ──
cfg['video_generation']['provider'] = 'ltx-video'

# cpu_offload DOIT rester True même sur A100-40Go :
# le modèle LTX-Video pèse ~28Go (T5-XXL 19Go + Transformer 7.7Go + VAE 1.7Go)
# → ne tient PAS en VRAM entier + activations = OOM + kill process
cfg['video_generation']['cpu_offload'] = True

cfg['video_generation']['generation']['num_inference_steps'] = 30
cfg['video_generation']['generation']['guidance_scale'] = 6.5
cfg['video_generation']['scenes']['max_scene_duration_sec'] = 10

# IMPORTANT: dimensions divisibles par 32 pour LTX-Video
# 768/32=24 ✓  1344/32=42 ✓  ratio=9:16 ✓
cfg['pipeline']['target_width'] = 768
cfg['pipeline']['target_height'] = 1344

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print('Config A100 appliquée ✓')
print(f"  Provider: {cfg['video_generation']['provider']}")
print(f"  Steps: {cfg['video_generation']['generation']['num_inference_steps']}")
print(f"  Resolution: {cfg['pipeline']['target_width']}x{cfg['pipeline']['target_height']} (div32 ✓)")
print(f"  cpu_offload: {cfg['video_generation']['cpu_offload']}")


## 6. Lancer le pipeline

In [ ]:
import os
TOPIC = "L'intelligence artificielle change le monde"
os.chdir('/content/SPARK')
!python pipeline.py --topic "{TOPIC}"

## 7. Télécharger le MP4 final

In [ ]:
import glob
from google.colab import files
renders = sorted(glob.glob('/content/SPARK/outputs/renders/*.mp4'))
if renders:
    final = renders[-1]
    print(f'Téléchargement: {final}')
    files.download(final)
else:
    print('Aucun fichier MP4 trouvé dans outputs/renders/')

## 8. (Optionnel) Sauvegarder les outputs sur Drive

In [ ]:
import shutil, os, glob
DRIVE_OUTPUTS = '/content/drive/MyDrive/SPARK_outputs'
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
for mp4 in glob.glob('/content/SPARK/outputs/renders/*.mp4'):
    dest = os.path.join(DRIVE_OUTPUTS, os.path.basename(mp4))
    shutil.copy2(mp4, dest)
    print(f'Sauvegardé → {dest}')